# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: 
* Model: 
* Evaluation approach: 
* Fine-tuning dataset: 

In [7]:
!pip install scikit-learn
import urllib.request
import csv
import pandas as pd
from transformers import (
    AutoTokenizer, DataCollatorWithPadding, TrainingArguments, Trainer,
    AutoModelForSequenceClassification
)
from peft import LoraConfig, get_peft_model
import numpy as np
from sklearn.metrics import accuracy_score
import torch
from torch.utils.data import Dataset

class AGNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding="max_length", max_length=256)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def load_ag_news():
    train_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
    test_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"
    
    urllib.request.urlretrieve(train_url, "train.csv")
    urllib.request.urlretrieve(test_url, "test.csv")
    
    def read_csv(filename):
        texts, labels = [], []
        with open(filename, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            for row in reader:
                labels.append(int(row[0]) - 1)
                texts.append(row[1])
        return texts, labels
    
    return map(read_csv, ["train.csv", "test.csv"])

# Load and prepare data
(train_texts, train_labels), (test_texts, test_labels) = load_ag_news()

# Use larger subset for better training
train_size = len(train_texts) // 100  # Changed from 500 to 100
test_size = len(test_texts) // 100    # Changed from 500 to 100

train_texts = train_texts[:train_size]
train_labels = train_labels[:train_size]
test_texts = test_texts[:test_size]
test_labels = test_labels[:test_size]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
train_dataset = AGNewsDataset(train_texts, train_labels, tokenizer)
test_dataset = AGNewsDataset(test_texts, test_labels, tokenizer)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

# Foundation Model
print("Evaluating foundation model...")
foundation_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

foundation_trainer = Trainer(
    model=foundation_model,
    args=TrainingArguments(
        output_dir="./results_foundation",
        per_device_eval_batch_size=16,
        remove_unused_columns=True
    ),
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

foundation_results = foundation_trainer.evaluate()

# PEFT Model with improved config
lora_config = LoraConfig(
    r=32,                    # Increased from 16
    lora_alpha=64,          # Increased from 32
    target_modules=["query", "key", "value", "output.dense"],  # Added output.dense
    lora_dropout=0.05,      # Reduced from 0.1
    bias="none",
    task_type="SEQ_CLS"
)

peft_model = get_peft_model(foundation_model, lora_config)
print(f"Trainable parameters: {peft_model.print_trainable_parameters()}")

# Improved training arguments
training_args = TrainingArguments(
    output_dir="./results_peft",
    num_train_epochs=5,              # Increased from 3
    per_device_train_batch_size=16,  # Increased from 8
    per_device_eval_batch_size=32,   # Increased from 16
    evaluation_strategy="steps",      # Changed from epoch
    eval_steps=50,                   # Added evaluation steps
    save_strategy="steps",
    save_steps=50,
    learning_rate=5e-4,              # Increased from 2e-5
    warmup_ratio=0.1,               # Added warmup
    weight_decay=0.01,              # Added weight decay
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True                       # Added mixed precision training
)

print("Training PEFT model...")
peft_trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

peft_trainer.train()
peft_results = peft_trainer.evaluate()

print("\nResults:")
print(f"Foundation model accuracy: {foundation_results['eval_accuracy']:.4f}")
print(f"PEFT model accuracy: {peft_results['eval_accuracy']:.4f}")
print(f"Improvement: {peft_results['eval_accuracy'] - foundation_results['eval_accuracy']:.4f}")

# Save final model
peft_trainer.save_model("bert-lora-ag-news-final")
tokenizer.save_pretrained("bert-lora-ag-news-final")

Evaluating foundation model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


trainable params: 3,840,008 || all params: 113,322,248 || trainable%: 3.3885737953239334
Trainable parameters: None
Training PEFT model...


Step,Training Loss,Validation Loss,Accuracy
50,No log,0.920436,0.684211
100,No log,0.575154,0.828947
150,No log,0.515383,0.842105
200,No log,0.451848,0.868421
250,No log,0.516877,0.881579
300,No log,0.503121,0.815789
350,No log,0.550542,0.828947



Results:
Foundation model accuracy: 0.2632
PEFT model accuracy: 0.8816
Improvement: 0.6184


('bert-lora-ag-news-final/tokenizer_config.json',
 'bert-lora-ag-news-final/special_tokens_map.json',
 'bert-lora-ag-news-final/vocab.txt',
 'bert-lora-ag-news-final/added_tokens.json',
 'bert-lora-ag-news-final/tokenizer.json')